In [1]:
from astropy.io import fits
from astropy.table import Table
#import matplotlib.pyplot as plt
import numpy as np
from pykoa.koa import Koa 
import os
import time
import pandas as pd
from pathlib import Path
import csv
import matplotlib.pyplot as plt


#!pip install pandas


In [2]:
## SHK index = alpha*((Nh + Nk)/(Nr + Nv))

## Nh and Nk are triangular weights centered around Ca II H + K lines with a 1.09 A FWHM

## Nr and Nv are rectangular weights with a width of 20 Å centred at 3901.07 Å and 4001.07 Å

## alpha is a historical scaling factor I need to investigate.

## It appears I might need to download more KOA files to get the Nr and Nv fluxes

In [3]:
width = 1.09
Hcenter_wave = 3933
Kcenter_wave = 3968

dflux = pd.read_csv('CHEMISTS/TOI-4638/HI.20221207.21074.93_1_04_flux.tbl.gz', sep='\s+') # or sep=',

waves = dflux.iloc[:,4]
flux = dflux.iloc[:,5]


#print(waves)

# Function that creates the triangular weight for each flux. max function makes it so that only a certain range of wavelengths have "weight"
def Triangular_Weight(width, center_wave, wave): 
    return max(0, 1 - (abs(wave - center_wave)/width))

HTri_Sum = 0

HWeighted_Flux = 0

# Loop to get the triangular sum for the H centered waves.
for i in range(len(waves)):
    HWeighted_Flux = flux[i]*Triangular_Weight(width, Hcenter_wave, waves[i])
    #print(HWeighted_Flux)
    HTri_Sum = HTri_Sum + HWeighted_Flux

print(HTri_Sum)


KTri_Sum = 0

KWeighted_Flux = 0

# Loop to get the triangular sum for the K centered waves.
for i in range(len(waves)):
    KWeighted_Flux = flux[i]*Triangular_Weight(width, Kcenter_wave, waves[i])
    #print(KWeighted_Flux)
    KTri_Sum = KTri_Sum + KWeighted_Flux

print(KTri_Sum)
    

3738.3880395683736
5595.986911829573


In [4]:
width2 = 10
Rcenter_wave = 3901.07
Vcenter_wave = 4001.07

dfluxR = pd.read_csv('Extra_Files/TOI-4638/HI.20221207.21074.93_1_03_flux.tbl.gz', sep='\s+') # or sep=',

dfluxV = pd.read_csv('Extra_Files/TOI-4638/HI.20221207.21074.93_1_05_flux.tbl.gz', sep='\s+') # or sep=',

wavesR = dfluxR.iloc[:,4]
fluxR = dfluxR.iloc[:,5]


wavesRsum = []
fluxRsum = []


# Loop to get all the flux values that belong to wavelengths that fall in our desired range
for i in range(len(wavesR)):
    if (Rcenter_wave - width2) <= wavesR[i] and wavesR[i] >= (Rcenter_wave + width2):
        wavesRsum.append(wavesR[i])
        fluxRsum.append(fluxR[i])

# Trapezoid rule for integration
Rsum = np.trapz(fluxRsum, wavesRsum)

print(Rsum)


wavesV = dfluxV.iloc[:,4]
fluxV = dfluxV.iloc[:,5]


wavesVsum = []
fluxVsum = []

# Loop to get all the flux values that belong to wavelengths that fall in our desired range
for i in range(len(wavesV)):
    if (Vcenter_wave - width2) <= wavesV[i] and wavesV[i] >= (Vcenter_wave + width2):
        wavesVsum.append(wavesV[i])
        fluxVsum.append(fluxV[i])

# Trapezoid rule for integration
Vsum = np.trapz(fluxVsum, wavesVsum)

print(Vsum)

7610.91079591468
8658.960277418737


In [5]:
#print(dfluxV.iloc[:,5])  ## Was used to find files that have the wavelengths I need

In [7]:
# Found S-value

#alpha value was found using a paper
alpha = 1.8

# The 8 and (1.09/20) were found from the "C3PO – IV. Co-natal stars depleted in refractories are magnetically more 
    # active – possible imprints of planets" paper

S = alpha * 8 * (1.09/20) * (HTri_Sum + KTri_Sum)/(Rsum + Vsum)

print(S)

0.45025663871817123
